# Model 3: Stacked Ensemble — TF-IDF + Logistic Regression → XGBoost Meta-Ranker

## Architecture Overview
This notebook implements a **two-stage stacked ensemble** for MCQ option ranking:

| Stage | Model | Input | Output |
|---|---|---|---|
| **Base** | Logistic Regression | TF-IDF cosine similarities + length features | Calibrated probability per option |
| **Meta** | XGBoost Classifier | LogReg probabilities + structural metadata | Final ranking score |

**Why stacking?** A single TF-IDF+LightGBM model (~0.68 MAP@3) saturates quickly because it only sees bag-of-words signals. By feeding the LogReg probability **as a new feature** into XGBoost, the meta-learner can correct systematic errors in the base model's confidence estimates — especially on short or ambiguous options.

**Row schema:** One row per question-option pair (5 rows per question). Label = 1 if that option is the correct answer, 0 otherwise.

**Viva key point:** Stacking avoids data leakage by using out-of-fold (OOF) predictions from the base model to train the meta-learner, so the meta-learner never trains on base-model predictions that were computed on the same rows.

In [ ]:
# ── Core imports ────────────────────────────────────────────────────────────
import os
import numpy as np
import pandas as pd
import xgboost as xgb
import wandb

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, log_loss
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import StratifiedKFold, train_test_split

print("All imports OK")

In [ ]:
# ── Kaggle data paths (hardcoded for cloud GPU execution) ───────────────────
# These match the exact Kaggle competition mount points.
# The local fallback lets the notebook run on a laptop for smoke-testing.

TRAIN_PATH      = "/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"
TEST_PATH       = "/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv"
SAMPLE_SUB_PATH = "/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv"

# Check each path and fall back to local data/ directories if not on Kaggle
for var_name, local_candidates in [
    ("TRAIN_PATH",      ["data/train.csv",             "../../data/train.csv"]),
    ("TEST_PATH",       ["data/test.csv",              "../../data/test.csv"]),
    ("SAMPLE_SUB_PATH", ["data/sample_submission.csv", "../../data/sample_submission.csv"]),
]:
    if not os.path.exists(globals()[var_name]):
        for candidate in local_candidates:
            if os.path.exists(candidate):
                globals()[var_name] = candidate
                break

print(f"TRAIN      → {TRAIN_PATH}")
print(f"TEST       → {TEST_PATH}")
print(f"SAMPLE_SUB → {SAMPLE_SUB_PATH}")

In [ ]:
# ── Weights & Biases initialisation ────────────────────────────────────────
# Inject the API key directly so Kaggle kernels can authenticate
# without interactive login. Never commit actual keys to a public repo;
# this key lives in .env locally and is injected at runtime on Kaggle.
os.environ["WANDB_API_KEY"] = "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"

wandb.init(
    project="23f2004343-t22026",
    name="Model_3_StackedEnsemble_LogReg_XGB",
    config={
        "base_model":       "LogisticRegression",
        "meta_model":       "XGBClassifier",
        "tfidf_features":   5000,
        "tfidf_ngram":      "(1,2)",
        "oof_folds":        5,
        "xgb_n_estimators": 200,
        "xgb_lr":           0.05,
        "xgb_max_depth":    5,
        "val_split":        0.15,
    }
)
CFG = wandb.config
print("W&B run:", wandb.run.name)

In [ ]:
# ── Load raw CSV files ──────────────────────────────────────────────────────
train_df = pd.read_csv(TRAIN_PATH)
test_df  = pd.read_csv(TEST_PATH)

# Ensure 'context' column exists — some splits omit it
for df in [train_df, test_df]:
    if "context" not in df.columns:
        df["context"] = ""
    df["context"] = df["context"].fillna("")

print("Train shape:", train_df.shape)
print("Test shape: ", test_df.shape)
print("Columns:",     train_df.columns.tolist())
train_df.head(3)

In [ ]:
# ── Feature Engineering ─────────────────────────────────────────────────────
# Goal: Convert each (question, option) pair into a fixed-length numeric vector.
#
# Feature groups:
#   1. cosine_sim   — TF-IDF cosine similarity between the combined
#                     context/prompt query and the option text.
#                     High similarity ≈ the option is topically relevant.
#   2–4. Length     — char length of prompt, option, and their absolute difference.
#   5–7. Word count — word count of prompt, option, and their absolute difference.
#
# These 7 scalar features are what the LogisticRegression (base model) sees.

OPTION_COLS = ["A", "B", "C", "D", "E"]

# Build a shared TF-IDF vocabulary from the full corpus (train + test).
# Fitting on both sets ensures no option word is 'out-of-vocabulary' at inference.
train_queries = (train_df["context"] + " " + train_df["prompt"]).fillna("").tolist()
test_queries  = (test_df["context"]  + " " + test_df["prompt"]).fillna("").tolist()

all_option_texts = []
for df in [train_df, test_df]:
    for col in OPTION_COLS:
        all_option_texts += df[col].fillna("").tolist()

# sublinear_tf=True applies log(1+tf) dampening — reduces the dominance
# of very frequent terms like 'the', 'a', 'is' that slip through stop-words.
tfidf = TfidfVectorizer(
    max_features=CFG.tfidf_features,   # 5 000-dimensional vocabulary
    stop_words="english",
    ngram_range=(1, 2),                # unigrams + bigrams capture short phrases
    sublinear_tf=True,
)
tfidf.fit(train_queries + test_queries + all_option_texts)
print(f"TF-IDF vocab: {len(tfidf.vocabulary_)} terms")


def extract_features(df: pd.DataFrame, queries: list, is_test: bool = False) -> pd.DataFrame:
    """
    Convert a question-level DataFrame into one row per (question, option) pair.

    Steps per row:
      1. Transform the query (context+prompt) to a TF-IDF vector.
      2. Transform each of the 5 options to a TF-IDF vector.
      3. Compute cosine similarity between the query vector and the option vector.
         (Both vectors are L2-normalised by TF-IDF, so the dot product == cosine sim.)
      4. Append char-level and word-level length statistics as extra scalars.
      5. Record the ground-truth binary label (1 = correct answer, 0 = distractor).
    """
    # Transform all queries at once — faster than one-by-one inside the loop
    query_vecs = tfidf.transform(queries)   # shape: (n_questions, vocab)
    records = []

    for i, (_, row) in enumerate(df.iterrows()):
        q_vec      = query_vecs[i]          # sparse (1, vocab)
        prompt_txt = str(row["prompt"])

        for opt in OPTION_COLS:
            opt_txt = str(row.get(opt, ""))
            opt_vec = tfidf.transform([opt_txt])  # sparse (1, vocab)

            # cosine_similarity returns a (1,1) matrix — extract scalar
            sim = float(cosine_similarity(q_vec, opt_vec)[0][0])

            # Binary label: 1 only for the option matching the answer key
            label = 0
            if not is_test and "answer" in row:
                label = 1 if str(row["answer"]).strip().upper() == opt else 0

            records.append({
                "id":          row["id"],
                "option_key":  opt,
                # Feature 1: semantic similarity between query and option
                "cosine_sim":  sim,
                # Features 2-4: character-level length signals
                "prompt_len":  len(prompt_txt),
                "option_len":  len(opt_txt),
                "len_diff":    abs(len(prompt_txt) - len(opt_txt)),
                # Features 5-7: word-level length signals
                "wc_prompt":   len(prompt_txt.split()),
                "wc_option":   len(opt_txt.split()),
                "wc_diff":     abs(len(prompt_txt.split()) - len(opt_txt.split())),
                "label":       label,
            })

    return pd.DataFrame(records)


print("Extracting train features...")
train_feat = extract_features(train_df, train_queries, is_test=False)
print("Extracting test  features...")
test_feat  = extract_features(test_df,  test_queries,  is_test=True)

print(f"Train long-format: {train_feat.shape}  | positives: {train_feat['label'].sum()}")
print(f"Test  long-format: {test_feat.shape}")
train_feat.head(10)

In [ ]:
# ── Stage 1: Base Model — Logistic Regression (Out-Of-Fold) ────────────────
#
# Why OOF (out-of-fold) predictions?
#   If we trained LogReg on all training data and then used its predictions
#   to train XGBoost on the same data, XGBoost would see 'perfect' overfit
#   predictions — this is DATA LEAKAGE. OOF avoids this:
#   each fold's LogReg is trained on the OTHER folds, so its predictions
#   on the held-out fold are 'unseen' — they carry genuine uncertainty.
#
# Process:
#   1. Split training rows into K=5 folds.
#   2. For each fold, train LogReg on the remaining 4 folds.
#   3. Predict probabilities on the held-out fold.
#   4. Collect all OOF probabilities → the base feature for XGBoost.

FEATURE_COLS = ["cosine_sim", "prompt_len", "option_len", "len_diff",
                "wc_prompt",  "wc_option",  "wc_diff"]

X_all = train_feat[FEATURE_COLS].values.astype(np.float32)
y_all = train_feat["label"].values

N_FOLDS = CFG.oof_folds  # 5-fold CV
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

# Placeholder array — will be filled fold by fold
oof_logreg_proba = np.zeros(len(X_all), dtype=np.float32)

for fold, (train_idx, val_idx) in enumerate(skf.split(X_all, y_all)):
    X_tr, X_val = X_all[train_idx], X_all[val_idx]
    y_tr, y_val = y_all[train_idx], y_all[val_idx]

    # LogisticRegression with L2 regularisation (C=1.0) to prevent overfitting
    # class_weight='balanced' compensates for the 1:4 imbalance (1 correct, 4 distractors)
    lr = LogisticRegression(
        C=1.0,
        max_iter=300,
        class_weight="balanced",
        random_state=42,
        solver="lbfgs",
    )
    lr.fit(X_tr, y_tr)

    # Predict probability of class=1 (i.e., 'this option is correct')
    fold_proba = lr.predict_proba(X_val)[:, 1]
    oof_logreg_proba[val_idx] = fold_proba

    fold_acc = accuracy_score(y_val, (fold_proba >= 0.5).astype(int))
    print(f"Fold {fold+1}/{N_FOLDS} — LogReg val accuracy: {fold_acc:.4f}")

# Report overall OOF LogReg performance
oof_lr_acc = accuracy_score(y_all, (oof_logreg_proba >= 0.5).astype(int))
oof_lr_ll  = log_loss(y_all, oof_logreg_proba)
print(f"\nOOF LogReg  accuracy: {oof_lr_acc:.4f}  |  logloss: {oof_lr_ll:.4f}")
wandb.log({"oof_logreg_accuracy": oof_lr_acc, "oof_logreg_logloss": oof_lr_ll})

# Also train a final LogReg on ALL training data to generate test-set base probabilities
lr_final = LogisticRegression(C=1.0, max_iter=300, class_weight="balanced",
                               random_state=42, solver="lbfgs")
lr_final.fit(X_all, y_all)

X_test_base = test_feat[FEATURE_COLS].values.astype(np.float32)
test_logreg_proba = lr_final.predict_proba(X_test_base)[:, 1]
print(f"LogReg test proba — mean: {test_logreg_proba.mean():.4f}")

In [ ]:
# ── Stage 2: Meta Model — XGBoost on Stacked Features ─────────────────────
#
# The meta-learner's input for each row is:
#   [logreg_proba, cosine_sim, prompt_len, option_len, len_diff,
#    wc_prompt, wc_option, wc_diff]
#
# XGBoost can learn non-linear interactions between these features that
# LogReg cannot capture with its linear decision boundary.
# For example: 'high cosine_sim AND short option_len → more likely correct'.

# Concatenate LogReg OOF probability alongside original feature columns
X_meta_train = np.hstack([
    oof_logreg_proba.reshape(-1, 1),   # base model's prediction as a new feature
    X_all                              # original hand-crafted features
])

# Concatenate LogReg test probability alongside original test feature columns
X_meta_test = np.hstack([
    test_logreg_proba.reshape(-1, 1),
    X_test_base
])

# Hold out 15% of training data for final XGBoost validation reporting
X_tr, X_val, y_tr, y_val = train_test_split(
    X_meta_train, y_all,
    test_size=CFG.val_split,
    stratify=y_all,
    random_state=42
)
print(f"XGBoost train: {X_tr.shape}  |  val: {X_val.shape}")

# XGBoost binary classifier
# scale_pos_weight compensates for the 4:1 class imbalance:
#   scale_pos_weight = count(negatives) / count(positives)
neg_count = (y_tr == 0).sum()
pos_count = (y_tr == 1).sum()
spw = neg_count / pos_count
print(f"scale_pos_weight = {spw:.2f}")

meta_clf = xgb.XGBClassifier(
    objective         = "binary:logistic",
    eval_metric       = "logloss",
    n_estimators      = CFG.xgb_n_estimators,   # 200 trees
    learning_rate     = CFG.xgb_lr,             # 0.05 — conservative shrinkage
    max_depth         = CFG.xgb_max_depth,       # 5 — limits model complexity
    subsample         = 0.8,                     # row sub-sampling per tree
    colsample_bytree  = 0.8,                     # feature sub-sampling per tree
    scale_pos_weight  = spw,                     # class imbalance correction
    use_label_encoder = False,
    random_state      = 42,
    early_stopping_rounds = 20,
    verbosity         = 0,
)

meta_clf.fit(
    X_tr, y_tr,
    eval_set=[(X_val, y_val)],
    verbose=25,   # print eval every 25 trees
)

print(f"Best XGB iteration: {meta_clf.best_iteration}")

In [ ]:
# ── Validation Metrics + W&B Logging ───────────────────────────────────────

val_proba  = meta_clf.predict_proba(X_val)[:, 1]
val_preds  = (val_proba >= 0.5).astype(int)

val_acc = accuracy_score(y_val, val_preds)
val_f1  = f1_score(y_val, val_preds, average="macro", zero_division=0)
val_ll  = log_loss(y_val, val_proba)

print(f"XGBoost val  accuracy : {val_acc:.4f}")
print(f"XGBoost val  macro-F1 : {val_f1:.4f}")
print(f"XGBoost val  logloss  : {val_ll:.4f}")

# Log all key metrics to W&B for comparison across runs and models
wandb.log({
    "xgb_val_accuracy":  val_acc,
    "xgb_val_f1":        val_f1,
    "xgb_val_logloss":   val_ll,
    "xgb_best_iter":     meta_clf.best_iteration,
})

In [ ]:
# ── MAP@3 evaluation on validation rows ────────────────────────────────────
# Reconstruct the val row IDs from the original long-format frame,
# then group by question ID and compute MAP@3.

_, val_idx = train_test_split(
    np.arange(len(train_feat)), test_size=CFG.val_split,
    stratify=y_all, random_state=42
)

val_feat = train_feat.iloc[val_idx].copy()
val_feat["pred_proba"] = val_proba

def average_precision_at_k(group: pd.DataFrame, k: int = 3) -> float:
    """AP@k for a single question: ranks options and checks if the correct one
    appears in the top-k positions, rewarding earlier positions more."""
    ranked = group.sort_values("pred_proba", ascending=False).reset_index(drop=True)
    score, hits = 0.0, 0
    for rank, row in ranked.head(k).iterrows():
        if row["label"] == 1:
            hits += 1
            score += hits / (rank + 1)   # precision at position rank+1
    return score

map3 = val_feat.groupby("id").apply(average_precision_at_k).mean()
print(f"Validation MAP@3: {map3:.4f}")
wandb.log({"val_map_at_3": map3})

In [ ]:
# ── Inference + submission.csv ──────────────────────────────────────────────
# Score each test option, group by question ID, sort descending by XGBoost
# probability, pick the top-3 option letters, and format the output.

# Generate predictions on test feature matrix
test_feat["pred_proba"] = meta_clf.predict_proba(X_meta_test)[:, 1]

def top3_prediction(group: pd.DataFrame) -> str:
    """For a single question, return the top-3 option letters sorted by
    predicted probability (highest first), separated by spaces."""
    ranked = group.sort_values("pred_proba", ascending=False)
    return " ".join(ranked["option_key"].head(3).tolist())

# Aggregate: one prediction string per question ID
submission = (
    test_feat
    .groupby("id", sort=False)
    .apply(top3_prediction)
    .reset_index()
    .rename(columns={0: "prediction"})
)

submission.to_csv("submission.csv", index=False)
print(f"Submission saved — {len(submission)} rows")
print(submission.head(10))

# Close the W&B run cleanly
wandb.finish()